# Sistema de clasificación de imágenes: Perros vs Gatos (Asirra)

Implementación paso a paso con `ImageDataGenerator`, `EfficientNetB0`, `ModelCheckpoint` y `EarlyStopping`.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd()))

from pipeline import (
    build_efficientnet_model,
    create_data_generators,
    evaluate_and_save_model,
    plot_first_nine_samples,
    prepare_dataset_splits,
    train_model,
)

BASE_DIR = Path.cwd().parent
RAW_TRAIN_DIR = BASE_DIR / "data" / "raw" / "dogs-vs-cats" / "train"
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"

## Paso 1 y Paso 2: Preparación del dataset y visualización de las primeras 9 fotos

In [ ]:
counts = prepare_dataset_splits(RAW_TRAIN_DIR, PROCESSED_DIR, test_size=0.2, seed=42)
print("Imágenes preparadas:", counts)

dogs_fig, cats_fig = plot_first_nine_samples(RAW_TRAIN_DIR, INTERIM_DIR, show=True)

In [ ]:
train_gen, test_gen = create_data_generators(
    PROCESSED_DIR,
    target_size=(224, 224),
    batch_size=32,
)

## Paso 3: Construcción de la RNA (EfficientNetB0)

In [ ]:
model = build_efficientnet_model(input_shape=(224, 224, 3), learning_rate=1e-3)
model.summary()

## Paso 4 y Paso 5: Optimización (`ModelCheckpoint` + `EarlyStopping`), evaluación y guardado

In [ ]:
best_model_path = MODELS_DIR / "efficientnet_b0_dogs_vs_cats_best.keras"
final_model_path = MODELS_DIR / "efficientnet_b0_dogs_vs_cats_final.keras"

history = train_model(
    model=model,
    train_generator=train_gen,
    test_generator=test_gen,
    checkpoint_path=best_model_path,
    epochs=10,
    patience=3,
)

metrics = evaluate_and_save_model(
    best_model_path=best_model_path,
    final_model_path=final_model_path,
    test_generator=test_gen,
)
print(f"Test Loss: {metrics['loss']:.4f} | Test Accuracy: {metrics['accuracy']:.4f}")
print(metrics["confusion_matrix"])
print(metrics["report"])